<a href="https://colab.research.google.com/github/HemanthSagarSR/BIS_LAB/blob/main/week5_grey_wolf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import numpy as np

# 1. Define the Cooling Goal (Objective Function)
def cooling_energy_cost(cooling_plan):
    """
    Calculates the energy cost score of a cooling configuration.
    Lesser energy cost score is better.

    cooling_plan[0] = Fan Speed Percentage (20% to 100%)
    cooling_plan[1] = AC Chiller Temperature Setpoint (18°C to 26°C)
    """
    fan_speed = cooling_plan[0]
    ac_temp = cooling_plan[1]

    # Base power cost model (higher fan speed and lower AC temps consume more power)
    fan_power = 0.05 * (fan_speed ** 2)
    ac_power = 1500 / (ac_temp - 10)  # Lower temps require exponentially more power
    total_power = fan_power + ac_power

    # Safety Check: Calculate server temperature under these settings
    # Server temp goes up if the AC is warm or if fans run too slowly
    estimated_server_temp = 15 + (35 - ac_temp) * 1.2 - (0.15 * fan_speed)

    # Severe penalty if the servers exceed the safe limit of 27.0°C
    if estimated_server_temp > 27.0:
        total_power += 5000 + (estimated_server_temp - 27.0) * 1000

    return total_power

# Main Grey Wolf Optimization Algorithm
def grey_wolf_optimizer_cooling(num_wolves=10, max_iter=20):
    # Search space limits: [Fan Speed %, AC Temp °C]
    lower_bounds = np.array([20.0, 18.0])
    upper_bounds = np.array([100.0, 26.0])
    dimensions = 2

    # 2. & 3. Generate Initial Random Cooling Plans (The Wolf Pack)
    positions = np.zeros((num_wolves, dimensions))
    for i in range(num_wolves):
        positions[i] = lower_bounds + random.random() * (upper_bounds - lower_bounds)

    # Initialize tracking variables for the top 3 winners
    alpha_pos, alpha_score = np.zeros(dimensions), float('inf')
    beta_pos, beta_score = np.zeros(dimensions), float('inf')
    delta_pos, delta_score = np.zeros(dimensions), float('inf')

    # Optimization Loop
    for t in range(max_iter):

        # 'a' decreases linearly from 2 down to 0 over time to slow down exploration
        a = 2.0 - t * (2.0 / max_iter)

        # 4. Score and 5. Find the Top 3 Winners
        for i in range(num_wolves):
            # Keep configuration settings within boundaries
            positions[i] = np.clip(positions[i], lower_bounds, upper_bounds)

            # Evaluate how well this setup performs
            score = cooling_energy_cost(positions[i])

            # Rank the leaders
            if score < alpha_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = alpha_score, alpha_pos.copy()
                alpha_score, alpha_pos = score, positions[i].copy()
            elif score < beta_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = score, positions[i].copy()
            elif score < delta_score:
                delta_score, delta_pos = score, positions[i].copy()

        # 6. Compare and 7. Nudge Remaining Settings Closer to Winners
        for i in range(num_wolves):
            for j in range(dimensions):

                # Math updates relative to Alpha wolf
                r1, r2 = random.random(), random.random()
                A1 = 2 * a * r1 - a
                C1 = 2 * r2
                D_alpha = abs(C1 * alpha_pos[j] - positions[i, j])
                X1 = alpha_pos[j] - A1 * D_alpha

                # Math updates relative to Beta wolf
                r1, r2 = random.random(), random.random()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2
                D_beta = abs(C2 * beta_pos[j] - positions[i, j])
                X2 = beta_pos[j] - A2 * D_beta

                # Math updates relative to Delta wolf
                r1, r2 = random.random(), random.random()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2
                D_delta = abs(C3 * delta_pos[j] - positions[i, j])
                X3 = delta_pos[j] - A3 * D_delta

                # Average out the guidance from the top three to find the wolf's next position
                positions[i, j] = (X1 + X2 + X3) / 3.0

        print(f"Iteration {t+1}/{max_iter} -> Current Lowest Power Cost Score: {alpha_score:.2f}")

    # 8. Return the Optimal Profile
    return alpha_pos, alpha_score

# Run the cooling simulation
best_cooling_plan, minimum_energy = grey_wolf_optimizer_cooling()

print("\n=== OPTIMAL COOLING PROFILE FOUND ===")
print(f"Optimized Fan Speed: {best_cooling_plan[0]:.2f}%")
print(f"Optimized AC Temperature Setpoint: {best_cooling_plan[1]:.2f}°C")
print(f"Minimized Power Cost Rating: {minimum_energy:.2f}")


Iteration 1/20 -> Current Lowest Power Cost Score: 255.36
Iteration 2/20 -> Current Lowest Power Cost Score: 254.19
Iteration 3/20 -> Current Lowest Power Cost Score: 162.02
Iteration 4/20 -> Current Lowest Power Cost Score: 162.02
Iteration 5/20 -> Current Lowest Power Cost Score: 117.35
Iteration 6/20 -> Current Lowest Power Cost Score: 113.75
Iteration 7/20 -> Current Lowest Power Cost Score: 113.75
Iteration 8/20 -> Current Lowest Power Cost Score: 113.75
Iteration 9/20 -> Current Lowest Power Cost Score: 113.75
Iteration 10/20 -> Current Lowest Power Cost Score: 113.75
Iteration 11/20 -> Current Lowest Power Cost Score: 113.75
Iteration 12/20 -> Current Lowest Power Cost Score: 113.75
Iteration 13/20 -> Current Lowest Power Cost Score: 113.75
Iteration 14/20 -> Current Lowest Power Cost Score: 113.75
Iteration 15/20 -> Current Lowest Power Cost Score: 113.75
Iteration 16/20 -> Current Lowest Power Cost Score: 113.75
Iteration 17/20 -> Current Lowest Power Cost Score: 113.75
Iterat